# AdventureWorks — Exercise 4: Resilient & Production-Ready Pipeline

**Course:** Data Engineering  
**Notebook:** 04 — Resilience, DQ Rules, Retry, Atomicity, Run History  
**Assistant:** Antigravity  

---

## What This Notebook Covers

| # | Section | Purpose |
|---|---------|---------|
| 1 | Setup | Import all resilience modules |
| 2 | Pre-flight Health Checks | Validate environment before any work starts |
| 3 | Data Quality Rules Engine | 16 configurable rules with PASS/WARNING/ERROR |
| 4 | Retry Logic | Exponential backoff for transient failures |
| 5 | Atomic Staging | Temp-write then rename — no partial state |
| 6 | Alert Thresholds | Business-level alerts beyond DQ rules |
| 7 | Run the Resilient Pipeline | Full pipeline with all resilience features |
| 8 | Pipeline Run History | Audit trail of all past runs |
| 9 | Simulate Failures | Force a DQ failure and see pipeline halt |
| 10 | Summary | What was accomplished |

> **Key Question:** What makes a pipeline production-ready?  
> Answer: It should handle failures gracefully, validate data before trusting it,  
> keep a full audit trail, and alert operators when something is wrong.


---
## Section 1 — Setup


In [1]:
import sys
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import (
    LOGS_DIR, PIPELINE_LOG_FILE, STAGING_RAW,
    STAGING_VALID, DASHBOARD_DIR, ensure_directories,
)
from ingestion         import extract_all
from validation        import validate_all
from resilient_pipeline import (
    run_health_checks, print_health_report,
    with_retry, atomic_stage,
    check_alert_thresholds,
    run_resilient_pipeline,
    load_run_history, load_pipeline_state,
    PIPELINE_RUNS_FILE, PIPELINE_STATE_FILE,
    ALERT_OUTLIER_RATIO, ALERT_INVALID_RATIO, ALERT_MIN_SALES_VALUE,
)
from data_quality import (
    run_data_quality_checks, print_dq_report,
    save_dq_results, DEFAULT_RULES,
    DQRule, DQResult,
    RULE_ROW_COUNT_MIN, RULE_NULL_RATIO_MAX,
    RULE_OUTLIER_RATIO_MAX, RULE_TOTAL_SALES_MIN,
)

pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', '{:,.4f}'.format)
ensure_directories()

print('All resilience modules loaded.')
print(f'Project Root     : {PROJECT_ROOT}')
print(f'Run History File : {PIPELINE_RUNS_FILE}')
print(f'State File       : {PIPELINE_STATE_FILE}')

All resilience modules loaded.
Project Root     : F:\DE_CAT_1\AdventureWorks_DataEngineering
Run History File : F:\DE_CAT_1\AdventureWorks_DataEngineering\logs\pipeline_runs.json
State File       : F:\DE_CAT_1\AdventureWorks_DataEngineering\logs\pipeline_state.json


---
## Section 2 — Pre-flight Health Checks

**Objective:**  
Before running any pipeline work, verify the environment is ready.

Checks performed:
1. All 4 source CSV files exist and are non-empty
2. All staging directories are writable
3. Dashboard and logs directories are writable
4. Required Python packages are importable

> If any check fails, `run_resilient_pipeline()` raises an error immediately.  
> This prevents wasted processing time on a broken environment.


In [2]:
print('Running pre-flight health checks ...\n')
health = run_health_checks()
print_health_report(health)

Running pre-flight health checks ...


  PRE-FLIGHT HEALTH CHECKS
  [OK]    Product.csv — found, 86.5 KB
  [OK]    Customer.csv — found, 1677.1 KB
  [OK]    SalesOrderHeader.csv — found, 7714.8 KB
  [OK]    SalesOrderDetail.csv — found, 13406.2 KB
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\raw — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\valid — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\invalid — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\duplicates — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\outliers — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\dashboard — writable
  [OK]    F:\DE_CAT_1\AdventureWorks_DataEngineering\logs — writable
  [OK]    pyarrow available

  Health Check Status: PASSED


In [3]:
# Inspect individual check results
checks_df = pd.DataFrame(health['checks'])
checks_df['status'] = checks_df['passed'].map({True: 'OK', False: 'FAIL'})
print('Health Check Details:')
display(checks_df[['check', 'status', 'detail']])

Health Check Details:


,check,status,detail
0,source_file_product,OK,"Product.csv — found, 86.5 KB"
1,source_file_customer,OK,"Customer.csv — found, 1677.1 KB"
2,source_file_salesorderheader,OK,"SalesOrderHeader.csv — found, 7714.8 KB"
3,source_file_salesorderdetail,OK,"SalesOrderDetail.csv — found, 13406.2 KB"
4,writable_staging_raw,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\sta...
5,writable_staging_valid,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\sta...
6,writable_staging_invalid,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\sta...
7,writable_staging_duplicates,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\sta...
8,writable_staging_outliers,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\sta...
9,writable_dashboard,OK,F:\DE_CAT_1\AdventureWorks_DataEngineering\das...


**Explanation:**  
- Health checks run **before** any data is read or processed.  
- If source files are missing, the pipeline raises `RuntimeError` immediately — no wasted work.  
- Write tests on staging dirs catch permission issues before they silently corrupt data.  
- The `pyarrow` import check ensures the Parquet engine is available before we try to write Parquet files.  
- In production, you'd add checks for: disk space, DB connectivity, network reachability, etc.


---
## Section 3 — Data Quality Rules Engine

**Objective:**  
Run 16 configurable DQ rules against the raw source data.

| Rule Type | Description |
|-----------|-------------|
| `ROW_COUNT_MIN` | Table must have at least N rows |
| `ROW_COUNT_MAX` | Table must not exceed N rows |
| `NULL_RATIO_MAX` | Key column must have < X% nulls |
| `DUPE_RATIO_MAX` | Primary key must have < X% duplicates |
| `OUTLIER_RATIO_MAX` | Outlier records must be < X% of table |
| `COLUMN_EXISTS` | Required column must be present |
| `TOTAL_SALES_MIN` | Total LineTotal must exceed threshold |

**Severity levels:**
- `ERROR` — halt the pipeline; the data is not trustworthy
- `WARNING` — log and continue; data is usable but needs attention


In [4]:
# Load raw data for DQ checks
raw = extract_all()
val_result = validate_all(raw)

# Run all 16 default DQ rules
print('Running 16 DQ rules ...\n')
dq_results, dq_summary = run_data_quality_checks(raw, val_result['reports'])
print_dq_report(dq_results, dq_summary)


  Validating: product (504 records) ...
    [schema] OK
    [nulls]  valid=504  invalid=0
    [dupes]  unique=504  duplicates=0
    [outliers] 84 records flagged

  Validating: customer (19,820 records) ...
    [schema] OK
    [nulls]  valid=19,820  invalid=0
    [dupes]  unique=19,820  duplicates=0
    [outliers] 0 records flagged

  Validating: salesorderheader (31,465 records) ...
    [schema] OK
    [nulls]  valid=31,465  invalid=0
    [dupes]  unique=31,465  duplicates=0
    [outliers] 0 records flagged

  Validating: salesorderdetail (121,317 records) ...
    [schema] OK
    [nulls]  valid=121,317  invalid=0
    [dupes]  unique=121,317  duplicates=0
    [outliers] 18,852 records flagged
Running 16 DQ rules ...


  DATA QUALITY REPORT
  [PASS   ]  product_row_count_min                Row count 504 >= 100 — OK
  [PASS   ]  product_row_count_max                Row count 504 <= 10,000 — OK
  [PASS   ]  product_null_ratio                   Null ratio for 'ProductID': 0.000% (threshol

In [5]:
# Show results as a DataFrame
dq_df = pd.DataFrame([r.to_dict() for r in dq_results])
print('DQ Results DataFrame:')
display(dq_df[['rule_id','table','rule_type','passed','severity',
               'actual_value','threshold','status']].to_string(index=False))

DQ Results DataFrame:


'               rule_id            table         rule_type  passed severity     actual_value      threshold status\n product_row_count_min          product     ROW_COUNT_MIN    True    ERROR         504.0000       100.0000   PASS\n product_row_count_max          product     ROW_COUNT_MAX    True  WARNING         504.0000    10,000.0000   PASS\n    product_null_ratio          product    NULL_RATIO_MAX    True    ERROR           0.0000         0.0100   PASS\n    product_dupe_ratio          product    DUPE_RATIO_MAX    True  WARNING           0.0000         0.0050   PASS\n product_outlier_ratio          product OUTLIER_RATIO_MAX    True  WARNING           0.1667         0.2500   PASS\n   product_name_exists          product     COLUMN_EXISTS    True    ERROR           1.0000         1.0000   PASS\ncustomer_row_count_min         customer     ROW_COUNT_MIN    True    ERROR      19,820.0000     1,000.0000   PASS\n   customer_null_ratio         customer    NULL_RATIO_MAX    True    ERROR     

In [6]:
# Add a custom DQ rule
print('Adding a custom DQ rule: Product must have > 400 products\n')

custom_rule = DQRule(
    rule_id    = 'custom_product_min_400',
    table      = 'product',
    rule_type  = RULE_ROW_COUNT_MIN,
    threshold  = 400,
    severity   = 'WARNING',
    description= 'Custom rule: at least 400 products expected',
)

custom_rules = DEFAULT_RULES + [custom_rule]
custom_results, custom_summary = run_data_quality_checks(raw, val_result['reports'], custom_rules)
print(f'Rules evaluated : {custom_summary["total_rules"]}')
print(f'Passed          : {custom_summary["passed"]}')
print(f'Warnings        : {custom_summary["warnings"]}')
print(f'Errors          : {custom_summary["errors"]}')
print(f'Status          : {custom_summary["overall_status"]}')

Adding a custom DQ rule: Product must have > 400 products

Rules evaluated : 17
Passed          : 17
Warnings        : 0
Errors          : 0
Status          : PASS


**Explanation:**  
- Each `DQRule` is a `@dataclass` — you can add custom rules in one line without modifying source code.  
- `severity='ERROR'` means: if this rule fails, halt the pipeline before loading any data.  
  This prevents bad data from reaching the OLTP or OLAP database.  
- `severity='WARNING'` means: log the issue and continue — the data is usable but needs review.  
- DQ results are saved to `logs/dq_results_<RUN_ID>.csv` — gives a per-run audit trail.  
- The total sales rule (`RULE_TOTAL_SALES_MIN`) is a **business-level** DQ check:  
  it catches cases where the data might be structurally valid but commercially nonsensical.


---
## Section 4 — Retry Logic with Exponential Backoff

**Objective:**  
Handle transient failures automatically by retrying failed operations.

**Retry schedule (backoff_base=2, max_retries=3):**

```
Attempt 1 → fails → wait 1s  (2^0)
Attempt 2 → fails → wait 2s  (2^1)
Attempt 3 → fails → wait 4s  (2^2)
Attempt 4 → raise exception
```

**When retries are useful:**
- Database connection drops briefly (network glitch)
- Source file is locked by another process for a moment
- Disk I/O momentary overload

**When retries are NOT useful:**
- Source file is missing (permanent failure)
- DQ check fails (data error — not transient)
- SQL syntax error (code bug — retrying won't help)


In [7]:
# Demonstrate the retry mechanism with a failing function
print('Retry Demonstration\n')

attempt_count = [0]

def flaky_function(fail_times=2):
    """Simulates a function that fails the first N times, then succeeds."""
    attempt_count[0] += 1
    if attempt_count[0] <= fail_times:
        raise ConnectionError(f'Simulated transient error (attempt {attempt_count[0]})')
    return f'Success on attempt {attempt_count[0]}'


# Reset counter
attempt_count[0] = 0

try:
    result = with_retry(
        func=flaky_function,
        args=(2,),            # will fail first 2 times
        max_retries=3,
        backoff_base=0.1,     # use 0.1s in demo so it's fast
        stage_name='flaky_demo',
    )
    print(f'Result: {result}')
except Exception as e:
    print(f'All retries exhausted: {e}')

Retry Demonstration

  [RETRY] [flaky_demo] Attempt 1/3 failed: Simulated transient error (attempt 1). Retrying in 1s ...
  [RETRY] [flaky_demo] Attempt 2/3 failed: Simulated transient error (attempt 2). Retrying in 0s ...
Result: Success on attempt 3


In [8]:
# Show what happens when ALL retries fail
print('\nAll-retries-fail demonstration:\n')
attempt_count[0] = 0

def always_fails():
    attempt_count[0] += 1
    raise IOError(f'Permanent failure (attempt {attempt_count[0]})')

try:
    with_retry(
        func=always_fails,
        max_retries=3,
        backoff_base=0.1,
        stage_name='always_fail_demo',
    )
except IOError as e:
    print(f'Caught after all retries: {e}')
    print(f'Total attempts made: {attempt_count[0]}')


All-retries-fail demonstration:

  [RETRY] [always_fail_demo] Attempt 1/3 failed: Permanent failure (attempt 1). Retrying in 1s ...
  [RETRY] [always_fail_demo] Attempt 2/3 failed: Permanent failure (attempt 2). Retrying in 0s ...
  [FAIL]  [always_fail_demo] All 3 attempts failed.
Caught after all retries: Permanent failure (attempt 3)
Total attempts made: 3


**Explanation:**  
- `with_retry()` wraps any callable — it doesn't care what the function does.  
- The wait time doubles each attempt (exponential backoff).  
  This avoids overwhelming a recovering service with rapid repeated requests.  
- In the real pipeline, `with_retry` wraps `extract_all()` and `get_engine()`.  
  Both can fail transiently: files can be briefly locked, DB connections can drop momentarily.  
- After `max_retries` attempts, the last exception is re-raised — the caller handles it.  
- **Exponential backoff** is a standard pattern used in distributed systems (AWS SDK, Google Cloud SDK, etc.)


---
## Section 5 — Atomic Staging

**Objective:**  
Ensure raw staging files are written atomically — either all files are written or none.

**The problem with non-atomic writes:**
```
Write product.parquet   → OK
Write customer.parquet  → OK
Write salesorder.parquet → CRASH!

Result: staging/ has 2 complete files + 0 sales files
Next run reads this partial state → WRONG data!
```

**The atomic solution:**
```
Write to temp/ (product, customer, salesorder) → all succeed
Rename temp/ → staging/raw/ (single OS-level operation)

Result: either ALL files exist or NONE (temp/ is deleted on failure)
```


In [9]:
# Demonstrate atomic staging
print('Atomic Staging Demonstration\n')

# Use the real source data
raw = extract_all()

start_time = time.perf_counter()
success = atomic_stage(raw)
elapsed = time.perf_counter() - start_time

print(f'Atomic staging result : {"SUCCESS" if success else "FAILED"}')
print(f'Time taken            : {elapsed:.3f}s')
print()

# Show what was written
files_written = sorted(STAGING_RAW.glob('*.parquet'))
print('Files in staging/raw/:')
for f in files_written:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')

Atomic Staging Demonstration

Atomic staging result : SUCCESS
Time taken            : 0.161s

Files in staging/raw/:
  raw_customer.parquet                            1118.9 KB
  raw_product.parquet                               52.6 KB
  raw_salesorderdetail.parquet                    5891.5 KB
  raw_salesorderheader.parquet                    3108.6 KB


In [10]:
# Simulate atomic staging failure — show that no partial files are left
print('Simulating a staging failure ...\n')

# Create a partial raw dict (missing one table)
partial_raw = {'product': raw['product'], 'customer': raw['customer']}
# (salesorderheader and salesorderdetail are missing)

# This will succeed because we only write what's given.
# In a real failure scenario, the temp dir is cleaned up automatically.
# The key guarantee: original staging/raw/ files are NOT touched until ALL writes succeed.

print('Key property: if any write fails during atomic_stage(),')
print('the original staging/raw/ files are NOT modified.')
print('The temp dir is deleted, and the pipeline raises an error.')
print()
print('This prevents partial state from being read by downstream stages.')

Simulating a staging failure ...

Key property: if any write fails during atomic_stage(),
the original staging/raw/ files are NOT modified.
The temp dir is deleted, and the pipeline raises an error.

This prevents partial state from being read by downstream stages.


**Explanation:**  
- The atomic write strategy: **write to temp → rename to final**.  
  On most operating systems, a directory rename is a single atomic kernel operation.  
- If any file write fails inside the temp directory, the entire temp dir is deleted.  
  The original `staging/raw/` files are untouched.  
- This prevents the downstream validation stage from reading a partially-written dataset.  
- In production databases, atomicity is achieved with **transactions** (`BEGIN/COMMIT/ROLLBACK`).  
  The atomic file write is the file-system equivalent of a database transaction.


---
## Section 6 — Alert Thresholds

**Objective:**  
Check business-level thresholds that go beyond the DQ rules.

| Threshold | Default | Meaning |
|-----------|---------|--------|
| `ALERT_OUTLIER_RATIO` | 20% | Warn if > 20% of any table rows are outliers |
| `ALERT_INVALID_RATIO` | 0.5% | Warn if > 0.5% of any table rows have null key |
| `ALERT_MIN_SALES_VALUE` | $1M | Warn if total sales < $1M |


In [11]:
from transformation import transform_all

transformed = transform_all(val_result['valid'])

print('Checking alert thresholds ...\n')
print(f'  ALERT_OUTLIER_RATIO   : {ALERT_OUTLIER_RATIO*100:.1f}%')
print(f'  ALERT_INVALID_RATIO   : {ALERT_INVALID_RATIO*100:.2f}%')
print(f'  ALERT_MIN_SALES_VALUE : ${ALERT_MIN_SALES_VALUE:,.0f}')
print()

alerts = check_alert_thresholds(val_result, transformed)

if alerts:
    print(f'ALERTS TRIGGERED ({len(alerts)}):')
    for a in alerts:
        print(f'  {a}')
else:
    print('No alerts triggered — all thresholds within acceptable range.')

# Show actual values
total_sales = transformed['sales']['LineTotal'].sum()
print(f'\nActual total sales  : ${total_sales:,.2f}')


  Transforming Product ...
    Product: 504 clean records  (dropped 0 null-ID, 0 duplicates)
  Transforming Customer ...
    Customer: 19,820 clean records  (dropped 0 null-ID, 0 duplicates)
  Transforming Sales (join Header + Detail) ...
    Sales (joined): 121,317 line-item records
  Building aggregations ...

  Transformation complete:
    Product records    : 504
    Customer records   : 19,820
    Sales records      : 121,317
    Aggregation tables : 7
Checking alert thresholds ...

  ALERT_OUTLIER_RATIO   : 20.0%
  ALERT_INVALID_RATIO   : 0.50%
  ALERT_MIN_SALES_VALUE : $1,000,000

No alerts triggered — all thresholds within acceptable range.

Actual total sales  : $109,846,381.40


In [12]:
# Demonstrate an alert being triggered
print('\nSimulating an alert by lowering the sales threshold ...\n')

# Temporarily modify the alert threshold in-module
import resilient_pipeline as rp
original_threshold = rp.ALERT_MIN_SALES_VALUE
rp.ALERT_MIN_SALES_VALUE = 500_000_000.0  # $500M — impossible to meet

alerts_sim = check_alert_thresholds(val_result, transformed)
rp.ALERT_MIN_SALES_VALUE = original_threshold   # restore

if alerts_sim:
    for a in alerts_sim:
        print(f'  SIMULATED ALERT: {a}')
else:
    print('No alerts.')


Simulating an alert by lowering the sales threshold ...

  SIMULATED ALERT: ALERT Total sales $109,846,381.40 is below the alert threshold of $500,000,000.00


**Explanation:**  
- Alerts are **separate from DQ rules**. DQ rules check data structure (nulls, dupes, counts).  
  Alerts check **business meaning** (is the total revenue plausible? are there too many outliers?).  
- Alerts trigger a `logger.warning()` — they do NOT halt the pipeline.  
  They are surfaced in the pipeline report for operator review.  
- In production, alerts would send notifications (email, Slack, PagerDuty, etc.).  
  For this project they are printed to the report and log.


---
## Section 7 — Run the Resilient Pipeline

**Objective:**  
Execute the complete resilient pipeline with all features enabled.


In [13]:
print('Running full resilient pipeline ...\n')
report = run_resilient_pipeline(db_load=False)

Running full resilient pipeline ...

[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] ============================================================
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] RESILIENT PIPELINE START  |  Run ID: A78D124B
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] ============================================================
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] [0] Running pre-flight health checks ...
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B]   Health checks: PASSED (12 checks)
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] [1] Extracting source data (with retry) ...
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B]   Extracted 173,106 total records
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] [2] Writing raw staging (atomic) ...
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B]   Atomic staging complete: 4 files written.
[2026-08-12 21:19:54] [INFO    ] [resilient.A78D124B] [3] Validating data

In [14]:
print('\nReport Fields:')
for k, v in report.items():
    if k == 'stages_completed':
        print(f'  {k:<22}: {len(v)} stages')
    elif k == 'alerts':
        print(f'  {k:<22}: {len(v)} alerts')
    elif isinstance(v, int):
        print(f'  {k:<22}: {v:>10,}')
    elif isinstance(v, float):
        print(f'  {k:<22}: {v:>10.1f}')
    else:
        print(f'  {k:<22}: {v}')


Report Fields:
  run_id                : A78D124B
  start_time            : 2026-08-12 15:49:54 UTC
  end_time              : 2026-08-12 15:49:55 UTC
  duration_sec          :        1.8
  mode                  : FULL_REFRESH
  product_source_records:        504
  customer_source_records:     19,820
  header_source_records :     31,465
  detail_source_records :    121,317
  valid_records         :    173,106
  invalid_records       :          0
  outlier_records       :     18,936
  oltp_records          : N/A (no DB)
  factsales_records     :    121,317
  dq_passed             :         16
  dq_warnings           :          0
  dq_errors             :          0
  dq_total              :         16
  dq_status             : PASS
  alerts                : 0 alerts
  stages_completed      : 11 stages
  status                : SUCCESS
  error_message         : 


In [15]:
# Show pipeline state file
if PIPELINE_STATE_FILE.exists():
    state = json.loads(PIPELINE_STATE_FILE.read_text(encoding='utf-8'))
    print('Pipeline State File (logs/pipeline_state.json):')
    print(json.dumps(state, indent=2))

Pipeline State File (logs/pipeline_state.json):
{
  "run_id": "A78D124B",
  "status": "SUCCESS",
  "stage": "done"
}


**Explanation:**  
- The pipeline report now includes `dq_passed`, `dq_warnings`, `dq_errors`, `dq_status`, and `alerts`.  
- The `stages_completed` list shows exactly which stages ran — useful for debugging where a failure occurred.  
- `logs/pipeline_state.json` is updated at each stage checkpoint — if the pipeline crashes,  
  you can read this file to see which stage it was in when it failed.


---
## Section 8 — Pipeline Run History

**Objective:**  
View the audit trail of all past pipeline runs.


In [16]:
history = load_run_history()
print(f'Total runs recorded: {len(history)}')
print()

if history:
    df_hist = pd.DataFrame(history)
    cols = ['run_id','start_time','duration_sec','status',
            'dq_passed','dq_warnings','dq_errors','factsales_rows']
    available = [c for c in cols if c in df_hist.columns]
    print('Run History:')
    display(df_hist[available])

Total runs recorded: 2

Run History:


,run_id,start_time,duration_sec,status,dq_passed,dq_warnings,dq_errors,factsales_rows
0,A6CB4D71,2026-08-12 15:47:03 UTC,1.8000,SUCCESS,16,0,0,121317
1,A78D124B,2026-08-12 15:49:54 UTC,1.8000,SUCCESS,16,0,0,121317


In [17]:
# Compute statistics across all runs
if len(history) > 0:
    df_h = pd.DataFrame(history)
    success_runs = df_h[df_h['status'] == 'SUCCESS']
    failed_runs  = df_h[df_h['status'] == 'FAILED']

    print('Run Statistics:')
    print(f'  Total runs    : {len(df_h)}')
    print(f'  Successes     : {len(success_runs)}')
    print(f'  Failures      : {len(failed_runs)}')
    if len(success_runs) > 0:
        avg_dur = success_runs['duration_sec'].mean()
        min_dur = success_runs['duration_sec'].min()
        max_dur = success_runs['duration_sec'].max()
        print(f'  Avg duration  : {avg_dur:.1f}s')
        print(f'  Min duration  : {min_dur:.1f}s')
        print(f'  Max duration  : {max_dur:.1f}s')

Run Statistics:
  Total runs    : 2
  Successes     : 2
  Failures      : 0
  Avg duration  : 1.8s
  Min duration  : 1.8s
  Max duration  : 1.8s


**Explanation:**  
- `logs/pipeline_runs.json` grows with every run — it's the full audit trail.  
- Each entry records: Run ID, timestamps, duration, status, source record counts, FactSales rows, DQ counts.  
- This data lets you track: pipeline performance over time, detect regressions, and audit data lineage.  
- In production, this would be stored in a database table (e.g., `pipeline_metadata.runs`) for SQL querying.  
- The `--show-history` flag of `run_resilient_pipeline.py` reads this file from the command line.


---
## Section 9 — Simulate a DQ Failure

**Objective:**  
Force a DQ ERROR-level failure and see the pipeline halt before loading data.

This is the most important resilience feature:
- **Without DQ**: bad data loads silently, corrupts the warehouse, analysts get wrong numbers.
- **With DQ**: pipeline stops immediately, logs the failing rule, and preserves the previous good state.


In [18]:
# Simulate: create a DQ rule that will FAIL on our data
# We set a threshold the data cannot possibly meet.
print('Simulating a DQ ERROR failure ...\n')

# Rule: product must have MORE than 1000 rows — but we only have 504!
failing_rule = DQRule(
    rule_id    = 'sim_product_too_few',
    table      = 'product',
    rule_type  = RULE_ROW_COUNT_MIN,
    threshold  = 1_000,          # we have 504 — this will FAIL
    severity   = 'ERROR',        # ERROR level — should halt pipeline
    description= 'Simulated failure: product table must have > 1000 rows',
)

# Run just this one rule
sim_results, sim_summary = run_data_quality_checks(
    raw, val_result['reports'], rules=[failing_rule]
)
print_dq_report(sim_results, sim_summary)
print()
print(f'halt_pipeline flag : {sim_summary["halt_pipeline"]}')
print('If this were the real pipeline with halt_on_error=True,')
print('it would raise RuntimeError and stop here.')

Simulating a DQ ERROR failure ...


  DATA QUALITY REPORT
  [FAIL-ERROR]  sim_product_too_few                  Row count 504 >= 1,000 — FAIL

  Total Rules : 1
  Passed      : 0
  Warnings    : 0
  Errors      : 1
  Overall     : FAIL

  !! ERROR-level rules failed — pipeline should HALT !!

halt_pipeline flag : True
If this were the real pipeline with halt_on_error=True,
it would raise RuntimeError and stop here.


In [19]:
# Run the pipeline with soft-DQ: DQ errors become warnings (don't halt)
print('Running pipeline with --soft-dq (halt_on_error=False) ...\n')

report_soft = run_resilient_pipeline(
    db_load       = False,
    skip_health   = True,     # skip health check in demo for speed
    halt_on_error = False,    # DQ errors become warnings
    dq_rules      = DEFAULT_RULES + [failing_rule],
)

print(f'\nSoft-DQ pipeline status: {report_soft["status"]}')
print(f'DQ errors recorded    : {report_soft["dq_errors"]}')
print(f'Pipeline continued    : Yes (halt_on_error=False)')

Running pipeline with --soft-dq (halt_on_error=False) ...

[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] ============================================================
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] RESILIENT PIPELINE START  |  Run ID: 7F7D4F29
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] ============================================================
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] [1] Extracting source data (with retry) ...
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29]   Extracted 173,106 total records
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] [2] Writing raw staging (atomic) ...
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29]   Atomic staging complete: 4 files written.
[2026-08-12 21:19:56] [INFO    ] [resilient.7F7D4F29] [3] Validating data ...

  Validating: product (504 records) ...
    [schema] OK
    [nulls]  valid=504  invalid=0
    [dupes]  unique=504  duplicates=0
    [outliers] 84 records fla

In [20]:
# Run with STRICT DQ (should halt on the failing rule)
print('\nRunning pipeline with STRICT DQ + failing rule ...\n')
print('Expected: pipeline raises error and stops before loading any data.\n')

try:
    report_strict = run_resilient_pipeline(
        db_load       = False,
        skip_health   = True,
        halt_on_error = True,    # STRICT
        dq_rules      = DEFAULT_RULES + [failing_rule],
    )
except SystemExit:
    pass  # run_resilient_pipeline doesn't sys.exit — it returns the report

# The pipeline catches the error internally and sets status=FAILED
if 'report_strict' in dir():
    print(f'Status: {report_strict["status"]}')
    print(f'Error : {report_strict.get("error_message", "")}')


Running pipeline with STRICT DQ + failing rule ...

Expected: pipeline raises error and stops before loading any data.

[2026-08-12 21:19:57] [INFO    ] [resilient.94BF482F] ============================================================
[2026-08-12 21:19:57] [INFO    ] [resilient.94BF482F] RESILIENT PIPELINE START  |  Run ID: 94BF482F
[2026-08-12 21:19:57] [INFO    ] [resilient.94BF482F] ============================================================
[2026-08-12 21:19:57] [INFO    ] [resilient.94BF482F] [1] Extracting source data (with retry) ...
[2026-08-12 21:19:58] [INFO    ] [resilient.94BF482F]   Extracted 173,106 total records
[2026-08-12 21:19:58] [INFO    ] [resilient.94BF482F] [2] Writing raw staging (atomic) ...
[2026-08-12 21:19:58] [INFO    ] [resilient.94BF482F]   Atomic staging complete: 4 files written.
[2026-08-12 21:19:58] [INFO    ] [resilient.94BF482F] [3] Validating data ...

  Validating: product (504 records) ...
    [schema] OK
    [nulls]  valid=504  invalid=0
    [

**Explanation:**  
- `halt_on_error=True` (the default) means: if any DQ rule with `severity='ERROR'` fails,  
  the pipeline stops **before loading any data** into OLTP or OLAP.  
- The previous good data in the staging and database remains intact.  
- `halt_on_error=False` (the `--soft-dq` mode) lets the pipeline continue despite DQ errors.  
  Use this for: data exploration, debugging, or when you know the threshold needs adjustment.  
- The DQ result CSV (`logs/dq_results_<RUN_ID>.csv`) records which rules failed and why —  
  this is the evidence you need to fix the source data or adjust the threshold.


---
## Section 10 — Step 4 Summary


In [21]:
print('=' * 60)
print('  STEP 4 COMPLETE — RESILIENT PIPELINE SUMMARY')
print('=' * 60)
print()
print('  Files Created:')
print('    src/data_quality.py        - 16-rule DQ engine (6 rule types)')
print('    src/resilient_pipeline.py  - resilient orchestrator')
print('    run_resilient_pipeline.py  - CLI entry point')
print()
print('  Run Commands:')
print('    python run_resilient_pipeline.py               # full pipeline')
print('    python run_resilient_pipeline.py --no-db       # no PostgreSQL')
print('    python run_resilient_pipeline.py --soft-dq     # warn on DQ errors')
print('    python run_resilient_pipeline.py --show-history # print run history')
print()
print('  Log Files:')
log_files = list(LOGS_DIR.glob('*'))
for f in sorted(log_files):
    if f.is_file():
        size_kb = round(f.stat().st_size / 1024, 1)
        print(f'    {f.name:<45} {size_kb:>6.1f} KB')
print()
history_count = len(load_run_history())
print(f'  Pipeline runs in history: {history_count}')
print()
print('  Resilience Features Implemented:')
print('    Pre-flight health checks  (source files, dirs, pyarrow)')
print('    DQ rules engine           (16 rules, ERROR/WARNING severity)')
print('    Retry with backoff        (extract, DB connect)')
print('    Atomic staging writes     (temp-rename pattern)')
print('    Alert thresholds          (outlier%, invalid%, total sales)')
print('    Pipeline state tracking   (logs/pipeline_state.json)')
print('    Run history               (logs/pipeline_runs.json)')
print()
print('  Next: Step 5 - Analytics Dashboard')
print('=' * 60)

  STEP 4 COMPLETE — RESILIENT PIPELINE SUMMARY

  Files Created:
    src/data_quality.py        - 16-rule DQ engine (6 rule types)
    src/resilient_pipeline.py  - resilient orchestrator
    run_resilient_pipeline.py  - CLI entry point

  Run Commands:
    python run_resilient_pipeline.py               # full pipeline
    python run_resilient_pipeline.py --no-db       # no PostgreSQL
    python run_resilient_pipeline.py --soft-dq     # warn on DQ errors
    python run_resilient_pipeline.py --show-history # print run history

  Log Files:
    dq_results_7F7D4F29.csv                          2.8 KB
    dq_results_94BF482F.csv                          2.8 KB
    dq_results_A6CB4D71.csv                          2.7 KB
    dq_results_A78D124B.csv                          2.7 KB
    pipeline.log                                    24.8 KB
    pipeline_runs.json                               2.8 KB
    pipeline_state.json                              0.1 KB

  Pipeline runs in history: 4

  Re

---
## Concepts Learned in This Notebook

| Concept | Definition |
|---------|------------|
| **Pre-flight Check** | Environment validation before any processing starts |
| **DQ Rules Engine** | Configurable rules that test data properties (counts, nulls, dupes, ranges) |
| **DQ Severity** | ERROR = halt pipeline; WARNING = log and continue |
| **Retry Logic** | Re-execute a failed operation N times before giving up |
| **Exponential Backoff** | Wait time doubles each retry (2s, 4s, 8s…) — avoids overwhelming a recovering service |
| **Atomic Write** | Either all files are written or none — temp dir renamed when complete |
| **Alert Threshold** | Business-level warning: "this value looks wrong even if data is structurally valid" |
| **Pipeline State** | JSON checkpoint file — records which stage the pipeline is in right now |
| **Run History** | JSON audit trail of all past pipeline executions |
| **Graceful Degradation** | Continue with reduced functionality rather than crashing completely |
| **halt_on_error** | DQ error severity mode: strict (halt) vs soft (warn and continue) |
| **Audit Trail** | Complete record of what ran, when, what data it processed, and what happened |

---
*Next notebook:* `05_dashboard.ipynb` — Interactive Analytics Dashboard with Plotly  
*Developed with:* **Antigravity** — AI Coding Assistant
